🔥 Day 3 Challenge 1

Now you write the code.

Using orders_df, answer:

Find the total quantity of products sold for each product.

Expected result:

Laptop  → 3
Mobile  → 2
Tablet  → 5

Requirements:

Use groupBy()
Use agg()
Use sum()
Give the result column the name total_quantity

In [0]:
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName("Day-3").getOrCreate()
print(spark)


In [0]:
orders_data = [
    ("O101", "C001", "Laptop", 1500, 2),
    ("O102", "C002", "Mobile", 800, 1),
    ("O103", "C001", "Mobile", 1200, 1),
    ("O104", "C003", "Tablet", 500, 3),
    ("O105", "C002", "Laptop", 2000, 1),
    ("O106", "C001", "Tablet", 700, 2),
]

orders_columns = [
    "order_id",
    "customer_id",
    "product",
    "amount",
    "quantity"
]

order_df=spark.createDataFrame(
    orders_data,
    orders_columns
)
order_df.show()

In [0]:
order_df.printSchema()

In [0]:
from pyspark.sql.functions import sum,count,avg,lit,coalesce

In [0]:
order_df.groupBy("product").agg(
    sum("quantity").alias("total_quantity")
).show()

### Challenge 2 — Total Revenue by Customer

Now solve this yourself:

Find the total revenue generated by each customer.

Use:

groupBy()
agg()
Spark sum()
alias("total_revenue")

In [0]:
order_df.show()

In [0]:
order_df.groupBy("customer_id").agg(
    sum("amount").alias("total_revenue")
).show()

### Challenge 3 — Multiple Aggregations

Now let's make it more realistic.

Business requirement:

For each customer, calculate:

- total_revenue → SUM of amount
- order_count → COUNT of orders
- avg_order_value → AVG of amount

In [0]:
order_df.groupBy("customer_id").agg(
    sum("amount").alias("total_revenue"),
    count("order_id").alias("order_count"),
    avg("amount").alias("avg_order_value")

).show()

### 🔥 Challenge 4 — JOIN

Now we move from aggregation → joins.

You already have:

- order_df
- order_id | customer_id | product | amount | quantity
- customer_df
- customer_id | customer_name | city
- Business requirement

Join the orders with customers and show the customer name along with every order.

Expected columns:

- order_id
- customer_id
- customer_name
- city
- product
- amount
- quantity

In [0]:
customers_data = [
    ("C001", "Shivam", "Delhi"),
    ("C002", "Rahul", "Mumbai"),
    ("C003", "Priya", "Bangalore"),
    ("C004", "Amit", "Noida"),
    ("C005", "Neha", "Gurgaon"),
]

customers_columns = [
    "customer_id",
    "customer_name",
    "city"
]

customer_df=spark.createDataFrame(customers_data,customers_columns)

In [0]:
joined_df=order_df.join(customer_df,
              on="customer_id",
              how='inner'
              )

joined_df.show()

🔥 Challenge 5 — LEFT JOIN

Now let's test whether you actually understand the difference between INNER and LEFT.

Imagine the customer table has a new customer who has never placed an order:

`customer_data = [
    ("C001", "Shivam", "Delhi"),
    ("C002", "Rahul", "Mumbai"),
    ("C003", "Priya", "Bangalore"),
    ("C004", "Amit", "Noida"),
    ("C005", "Neha", "Gurgaon")
]`

C005 has no order.

Business requirement

Show ALL customers, even if they have never placed an order.

In [0]:
join_df=customer_df.join(order_df,
                 on="customer_id",
                 how="left")


join_df.show()

### 🔥 Challenge 6 — JOIN + AGGREGATION

Now we're going to combine the two concepts you've learned.

Business requirement

Show total revenue for each customer, including customers who have never placed an order.

In [0]:
customer_df.join(order_df,on="customer_id",how="left").groupBy("customer_id").agg(
    sum("amount").alias("total_revenue")
).show()

Challenge 7 — GROUP BY + JOIN + multiple metrics
Business requirement

"For every customer, show their name, city, number of orders, total revenue, and average order value. Include customers who have never placed an order."

Expected structure:

`customer_id | customer_name | city | order_count | total_revenue | avg_order_value`

For customers with no orders, they should still appear.

Your task

Use:

- customer_df
- order_df
- LEFT JOIN
- groupBy()
- agg()
- count()
- sum()
- avg()
- appropriate aliases

In [0]:
customer_df.join(order_df,on="customer_id",how="left").groupBy(["customer_id","city","customer_name"]).agg(
    coalesce(count("order_id"),lit(0)).alias("order_count"),
    coalesce(sum("amount"),lit(0)).alias("total_revenue"),
    coalesce(avg("amount"),lit(0)).alias("avg_order_value")

).show()

Challenge 8 — RIGHT JOIN vs LEFT JOIN

Now let's test whether you've understood which table is preserved.

Business requirement

**"Show every order and the customer information if available. Even if an order has no matching customer."**

Imagine order_df contains:

- O101 → C001
- O102 → C002
- O103 → C001
- O104 → C003
- O105 → C002
- O106 → C001
- O107 → C999   ← customer doesn't exist
Your task

Write PySpark code that:

preserves every order
brings customer_name and city when available
uses a join
displays:
order_id
customer_id
product
customer_name
city

Think carefully about:

Which DataFrame should be on the LEFT side?

In [0]:
order_df.join(customer_df,on="customer_id",how="left").show()

Challenge 9 — FULL OUTER JOIN

Now let's test the final major join type.

Business requirement

"Show all customers and all orders, even when there is no match between them."

Meaning:

- Customers with orders → show both
- Customers without orders → still show customer
- Orders without customers → still show order

Use:

- customer_df
- order_df


Write the PySpark code using:

- join()
- on="customer_id"
- how="full" or "full_outer"
- .select() to show only:
- customer_id
- customer_name
- city
- order_id
- product
- amount



In [0]:
order_df.join(customer_df,on="customer_id",how="full").select(
    "customer_id",
    "customer_name",
    "city",
    "order_id",
    "product",
    "amount").show()

Challenge 10 — Real business problem

Now let's combine what you've learned.

Business requirement:

**Find the top 2 customers by total revenue.
**
- Use order_df and:
- groupBy()
- sum()
- alias()
- orderBy()
- desc()
- limit()

In [0]:
order_df.show()

In [0]:
order_df.groupBy("customer_id").agg(
    sum("amount").alias("total_revenue")).orderBy("total_revenue",ascending=False).limit(2).show()

PySpark Aggregations & Joins
Question 11/15 — HAVING Equivalent
🏢 Business Problem

The business team says:

**"Show only customers who generated more than 2,000 in total revenue."**

We already have:

order_df

with these columns:

- order_id
- customer_id
- product
- amount
- quantity
🎯 Your Task

**Write PySpark code that:**

Groups the data by `customer_id`
Calculates total amount for each customer
Names the result total_revenue
Keeps only customers where total_revenue > 2000
Displays the result

In [0]:
order_df.show()

In [0]:
order_df.groupBy("customer_id").agg(
    sum("amount").alias("total_revenue")
).filter("total_revenue>2000").show()

In [0]:
from pyspark.sql.functions import col,window

🔥 Question 12/15 — Join + Aggregation + Filtering
🏢 Business Problem

The business team wants to know:

"Which cities generated more than ₹2,500 in total revenue?"

You have:

 order_df
- order_id
- customer_id
- product
- amount
- quantity

customer_df

- customer_id
- customer_name
- city

In [0]:
customer_df.join(order_df,on="customer_id",how="left").groupBy("city").agg(
    sum("amount").alias("total_revenue")
).filter(col("total_revenue")>2500).show()

**🔥 Question 13/15 — Multiple Grouping
🏢 Business Problem**

The business team now wants:

`Show total revenue for each product in each city.`

In [0]:
customer_df.join(order_df,on="customer_id",how="inner").groupBy(["city","product"]).agg(
    sum("amount").alias("total_revenue")
).show()

**🔥 Question 14/15 — Top Product Per Customer

Now we're going one level harder.**

🏢 Business Problem

The business team asks:

`For each customer, identify the product that generated the highest total revenue.`

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

In [0]:
product_revenue=customer_df.join(order_df,on="customer_id",how="inner").groupBy(["product","customer_id"]).agg(
    sum("amount").alias("total_revenue")
)
window_spec=Window.partitionBy("customer_id").orderBy(col("total_revenue").desc())

ranked_df=product_revenue.withColumn("rn",row_number().over(window_spec))

ranked_df.filter(col("rn")==1).show()

🔥 Question 15/15 — Final Day 3 Challenge

Business requirement:

For each customer, calculate their total revenue and order count, then identify customers whose total revenue is greater than 2500.

You need to use:

- customer_df
- order_df
- join()
- groupBy()
- sum()
- count()
- filter()

In [0]:
customer_df.join(order_df,on="customer_id",how="left").groupBy("customer_id").agg(
    sum("amount").alias("total_revenue"),
    count("order_id").alias("order_count")
).filter(col("total_revenue")>2500).show()